In [11]:
from pathlib import Path
import sys, json
import numpy as np
from scipy.optimize import curve_fit


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for c in (start, *start.parents):
        if (c / "src").is_dir() and (c / "requirements.txt").exists():
            return c
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "tests" / "results"
FIG_DIR = REPO_ROOT / "tests" / "figures"
DATA_DIR.mkdir(exist_ok=True, parents=True)
FIG_DIR.mkdir(exist_ok=True, parents=True)

from src.simulation.exact_simulator import ExactSimulator
from src.core.seeding import ExperimentSeeder
from src.circuits.families import FAMILIES
from src.analysis.results_io import TrialRecord, TrialLedger
from src.visualization.style import mplstyle
import matplotlib.pyplot as plt

mplstyle()

print("Imports OK.")
print("Registered families:", list(FAMILIES.keys()))

Imports OK.
Registered families: ['1D Brickwork', '2D Brickwork', 'RFIM (W=2)', 'QAOA (p=3)', 'UCCSD', 'Haar']


In [ ]:
# Configuration
QUICK_TEST = False  # flip to True first to sanity-check the pipeline quickly

EXPERIMENT_ENTROPY = 20260810  # log this -- it reproduces every seed below
N_BOOT = 4000

# Every registered family EXCEPT UCCSD.
SWEEP_FAMILY_NAMES = [name for name in FAMILIES if name != "UCCSD"]

if QUICK_TEST:
    N_VALUES = [8, 10]
    N_TRIALS = 5
else:
    N_VALUES = [10, 12, 14, 16, 18, 20]
    N_TRIALS = 100

exp = ExperimentSeeder(entropy=EXPERIMENT_ENTROPY)

NPZ_PATH = DATA_DIR / (
    "prz_scaling_fit_allfamilies_quick.npz"
    if QUICK_TEST
    else "prz_scaling_fit_allfamilies.npz"
)
META_PATH = NPZ_PATH.with_suffix(".meta.json")

EXPECTED_META = {
    "N_values": N_VALUES,
    "n_trials": N_TRIALS,
    "entropy": exp.entropy,
    "families": SWEEP_FAMILY_NAMES,
}


def compute_prz(sv):
    "Exact-state participation ratio: 1 / sum(|psi|^4)"
    p = np.abs(sv) ** 2
    return float(1.0 / np.sum(p**2))


print(f"Experiment entropy (log this): {exp.entropy}")
print(f"N_VALUES={N_VALUES}, N_TRIALS={N_TRIALS}")
print(f"Families swept ({len(SWEEP_FAMILY_NAMES)}): {SWEEP_FAMILY_NAMES}")
print(f"Total exact simulations: {len(SWEEP_FAMILY_NAMES) * len(N_VALUES) * N_TRIALS}")

Experiment entropy (log this): 20260810
N_VALUES=[10, 12, 14, 16, 18, 20], N_TRIALS=100
Families swept (5): ['1D Brickwork', '2D Brickwork', 'RFIM (W=2)', 'QAOA (p=3)', 'Haar']
Total exact simulations: 3000


In [ ]:
# Run (or load cached) sweep
# prz_by_family[family_name] has shape (N_TRIALS, len(N_VALUES)), matching
# the single-family version's array layout exactly, just one per family now.
# Every individual (family, N, trial) draw is also recorded into a
# TrialLedger for full per-trial, per-seed traceability.
def run_or_load():
    ledger = TrialLedger()

    if NPZ_PATH.exists() and META_PATH.exists():
        stored_meta = json.loads(META_PATH.read_text())
        if stored_meta == EXPECTED_META:
            print("Loaded from cache:", NPZ_PATH)
            raw = dict(np.load(NPZ_PATH, allow_pickle=False))
            prz_by_family = {name: raw[f"prz__{name}"] for name in SWEEP_FAMILY_NAMES}
            trials_csv = DATA_DIR / NPZ_PATH.name.replace(".npz", "_trials.csv")
            if trials_csv.exists():
                ledger = TrialLedger.from_csv(trials_csv)
            return prz_by_family, ledger
        print("Cache meta mismatch, re-running.")

    prz_by_family = {
        name: np.full((N_TRIALS, len(N_VALUES)), np.nan) for name in SWEEP_FAMILY_NAMES
    }

    for fam_name in SWEEP_FAMILY_NAMES:
        fam = FAMILIES[fam_name]
        print(f"\n[{fam_name}]")
        for ni, N in enumerate(N_VALUES):
            exact_sim = ExactSimulator(N, verbose=False)
            for t in range(N_TRIALS):
                seed = exp.seq(fam_name, N, t)
                rng = np.random.default_rng(seed)
                circuit = fam.generate(N, rng)
                sv = exact_sim.simulate(circuit)
                prz = compute_prz(sv)
                prz_by_family[fam_name][t, ni] = prz
                ledger.add(
                    TrialRecord(
                        family=fam_name,
                        method="exact",
                        N=N,
                        trial=t,
                        seed_entropy=exp.entropy,
                        extra={"prz": prz},
                    )
                )
            gm = float(np.exp(np.mean(np.log(prz_by_family[fam_name][:, ni]))))
            print(f"  N={N}: PRZ geomean = {gm:.1f}")

    save_arrs = {f"prz__{name}": arr for name, arr in prz_by_family.items()}
    np.savez(NPZ_PATH, **save_arrs)
    META_PATH.write_text(json.dumps(EXPECTED_META, indent=2))
    trials_csv = DATA_DIR / NPZ_PATH.name.replace(".npz", "_trials.csv")
    trials_npz = DATA_DIR / NPZ_PATH.name.replace(".npz", "_trials.npz")
    ledger.to_csv(trials_csv)
    ledger.to_npz(trials_npz)
    print("\nSaved to cache. Per-trial ledger:", len(ledger), "rows")
    return prz_by_family, ledger


prz_by_family, ledger = run_or_load()


[1D Brickwork]
  N=10: PRZ geomean = 329.4
  N=12: PRZ geomean = 1162.5
  N=14: PRZ geomean = 3786.5
  N=16: PRZ geomean = 13857.4
  N=18: PRZ geomean = 46439.8
  N=20: PRZ geomean = 158854.4

[2D Brickwork]
  N=10: PRZ geomean = 219.1
  N=12: PRZ geomean = 800.7
  N=14: PRZ geomean = 2313.8
  N=16: PRZ geomean = 8325.8
  N=18: PRZ geomean = 25524.3
  N=20: PRZ geomean = 90688.2

[RFIM (W=2)]
  N=10: PRZ geomean = 86.0
  N=12: PRZ geomean = 163.4
  N=14: PRZ geomean = 456.0
  N=16: PRZ geomean = 923.0
  N=18: PRZ geomean = 2244.4
  N=20: PRZ geomean = 5218.4

[QAOA (p=3)]
  N=10: PRZ geomean = 88.6
  N=12: PRZ geomean = 210.8
  N=14: PRZ geomean = 547.6
  N=16: PRZ geomean = 1277.3
  N=18: PRZ geomean = 3041.3
  N=20: PRZ geomean = 7324.0

[Haar]
  N=10: PRZ geomean = 241.9
  N=12: PRZ geomean = 724.9
  N=14: PRZ geomean = 2454.2
  N=16: PRZ geomean = 6884.8
  N=18: PRZ geomean = 24541.4
  N=20: PRZ geomean = 65576.5

Saved to cache. Per-trial ledger: 3000 rows


In [14]:
# Pool across families at each N, then fit ONE exponential curve
# pooled_samples[ni] = concatenation, across every swept family, of that
# family's N_TRIALS raw PRZ draws at N_VALUES[ni].
N_arr = np.array(N_VALUES, dtype=float)
pooled_samples = [
    np.concatenate([prz_by_family[name][:, ni] for name in SWEEP_FAMILY_NAMES])
    for ni in range(len(N_VALUES))
]
pooled_gmean = np.array([float(np.exp(np.mean(np.log(s)))) for s in pooled_samples])


def exp_model(N, alpha, logC):
    return logC + alpha * np.asarray(N, float) * np.log(2)


popt, pcov = curve_fit(exp_model, N_arr, np.log(pooled_gmean), p0=[0.70, 0.0])
alpha_fit, logC_fit = popt
C_fit = float(np.exp(logC_fit))

# Bootstrap CI: resample each N's pooled sample independently, with
# replacement, at its own original (pooled, cross-family) size.
rng_b = np.random.default_rng(exp.entropy)
boot_alphas = np.empty(N_BOOT)
for b in range(N_BOOT):
    boot_gm = np.array(
        [
            float(np.exp(np.mean(np.log(rng_b.choice(s, size=len(s), replace=True)))))
            for s in pooled_samples
        ]
    )
    try:
        bp, _ = curve_fit(exp_model, N_arr, np.log(boot_gm), p0=[alpha_fit, logC_fit])
        boot_alphas[b] = bp[0]
    except Exception:
        boot_alphas[b] = np.nan

alpha_lo = float(np.nanpercentile(boot_alphas, 2.5))
alpha_hi = float(np.nanpercentile(boot_alphas, 97.5))

print("=== Pooled fit across all swept families ===")
print(f"Families pooled: {SWEEP_FAMILY_NAMES}")
print(f"PRZ ~ {C_fit:.2f} * 2^({alpha_fit:.3f} * N)")
print(f"alpha = {alpha_fit:.3f}   95% CI [{alpha_lo:.3f}, {alpha_hi:.3f}]")
print()
print("For reference, manuscript per-family claims (Appendix D.3):")
print("  Brickwork L=5:  alpha = 0.70 +- 0.02")
print("  Haar L=3:       alpha = 0.60 +- 0.03")
print("  2D Brickwork:   alpha = 0.65 +- 0.04")
print()
print("Per-family geometric means at each N (for context; NOT separately fit here):")
for fam_name in SWEEP_FAMILY_NAMES:
    gms = [
        float(np.exp(np.mean(np.log(prz_by_family[fam_name][:, ni]))))
        for ni in range(len(N_VALUES))
    ]
    print(f"  {fam_name:16s}", ["%.1f" % g for g in gms])

np.savez(
    DATA_DIR / "prz_scaling_fit_allfamilies_result.npz",
    N_values=np.array(N_VALUES),
    pooled_gmean=pooled_gmean,
    alpha=alpha_fit,
    alpha_ci_lo=alpha_lo,
    alpha_ci_hi=alpha_hi,
    C=C_fit,
    entropy=exp.entropy,
    n_trials=N_TRIALS,
    families=np.array(SWEEP_FAMILY_NAMES),
)
print("\nResult saved to tests/results/prz_scaling_fit_allfamilies_result.npz")

=== Pooled fit across all swept families ===
Families pooled: ['1D Brickwork', '2D Brickwork', 'RFIM (W=2)', 'QAOA (p=3)', 'Haar']
PRZ ~ 0.85 * 2^(0.761 * N)
alpha = 0.761   95% CI [0.744, 0.779]

For reference, manuscript per-family claims (Appendix D.3):
  Brickwork L=5:  alpha = 0.70 +- 0.02
  Haar L=3:       alpha = 0.60 +- 0.03
  2D Brickwork:   alpha = 0.65 +- 0.04

Per-family geometric means at each N (for context; NOT separately fit here):
  1D Brickwork     ['329.4', '1162.5', '3786.5', '13857.4', '46439.8', '158854.4']
  2D Brickwork     ['219.1', '800.7', '2313.8', '8325.8', '25524.3', '90688.2']
  RFIM (W=2)       ['86.0', '163.4', '456.0', '923.0', '2244.4', '5218.4']
  QAOA (p=3)       ['88.6', '210.8', '547.6', '1277.3', '3041.3', '7324.0']
  Haar             ['241.9', '724.9', '2454.2', '6884.8', '24541.4', '65576.5']

Result saved to tests/results/prz_scaling_fit_allfamilies_result.npz
